<a href="https://colab.research.google.com/github/sinayavarian/ML-practice/blob/main/Sina_Task4_Movie_Recommenders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Title + Introduction

# Movie Recommender System - MovieLens 20M

This notebook implements multiple recommendation techniques:
- Popularity-Based
- Content-Based Filtering
- Collaborative Filtering
- Matrix Factorization (Surprise)
- Hybrid Model

In [2]:
!pip install scikit-surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl size=2554975 sha256=3b6a2c77af680f1ce2b3be41156dff2b5b0a342c0109dc722bad3b87446ed464
  Stored in directory: /root/.cache/pip/wheels/75/fa/bc/739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise


In [1]:
!pip uninstall -y numpy
!pip install "numpy<2"
!pip install scikit-surprise --no-cache-dir

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
^C


# 2. Import Libraries

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

print("Surprise imported successfully!")

NumPy version: 1.26.4
Surprise imported successfully!


In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

from scipy.sparse import csr_matrix

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# 3. Load Data

In [4]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/movielens', exist_ok=True)

In [ ]:
import shutil

shutil.move('/content/movies.csv', '/content/drive/MyDrive/movielens/movies.csv')
shutil.move('/content/ratings.csv', '/content/drive/MyDrive/movielens/ratings.csv')
shutil.move('/content/tags.csv', '/content/drive/MyDrive/movielens/tags.csv')

'/content/drive/MyDrive/movielens/tags.csv'

In [ ]:
os.listdir('/content/drive/MyDrive/movielens')

['movies.csv', 'ratings.csv', 'tags.csv']

In [4]:
path = '/content/drive/MyDrive/movielens/'

movies = pd.read_csv(path + 'movies.csv')
ratings = pd.read_csv(path + 'ratings.csv')
tags = pd.read_csv(path + 'tags.csv')

# 4. Data Overview

In [5]:
print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
print("Tags shape:", tags.shape)

print("\nMovies columns:", movies.columns.tolist())
print("Ratings columns:", ratings.columns.tolist())
print("Tags columns:", tags.columns.tolist())

display(movies.head())
display(ratings.head())
display(tags.head())

Movies shape: (27278, 3)
Ratings shape: (20000263, 4)
Tags shape: (465564, 4)

Movies columns: ['movieId', 'title', 'genres']
Ratings columns: ['userId', 'movieId', 'rating', 'timestamp']
Tags columns: ['userId', 'movieId', 'tag', 'timestamp']


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


,userId,movieId,tag,timestamp
0,18,4141,Mark Waters,1240597180
1,65,208,dark hero,1368150078
2,65,353,dark hero,1368150079
3,65,521,noir thriller,1368149983
4,65,592,dark hero,1368150078


In [6]:
print("Movies nulls:\n", movies.isnull().sum())
print("\nRatings nulls:\n", ratings.isnull().sum())
print("\nTags nulls:\n", tags.isnull().sum())

Movies nulls:
 movieId    0
title      0
genres     0
dtype: int64

Ratings nulls:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Tags nulls:
 userId        0
movieId       0
tag          16
timestamp     0
dtype: int64


# 5. Popularity Recommender

In [7]:
movie_stats = ratings.groupby('movieId').agg(
    rating_count=('rating', 'count'),
    rating_mean=('rating', 'mean')
).reset_index()

movie_stats = movie_stats.merge(movies, on='movieId', how='left')

print(movie_stats.shape)
display(movie_stats.head())


(26744, 5)


,movieId,rating_count,rating_mean,title,genres
0,1,49695,3.921240,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,22243,3.211977,Jumanji (1995),Adventure|Children|Fantasy
2,3,12735,3.151040,Grumpier Old Men (1995),Comedy|Romance
3,4,2756,2.861393,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,12161,3.064592,Father of the Bride Part II (1995),Comedy


In [8]:
C = movie_stats['rating_mean'].mean()
m = movie_stats['rating_count'].quantile(0.90)

print("C (overall mean rating):", C)
print("m (minimum votes threshold):", m)

C (overall mean rating): 3.1331999901256933
m (minimum votes threshold): 1305.7000000000007


In [9]:
qualified = movie_stats[movie_stats['rating_count'] >= m].copy()

qualified['weighted_rating'] = (
    (qualified['rating_count'] / (qualified['rating_count'] + m)) * qualified['rating_mean']
    + (m / (qualified['rating_count'] + m)) * C
)

qualified = qualified.sort_values('weighted_rating', ascending=False)

display(qualified[['movieId', 'title', 'rating_count', 'rating_mean', 'weighted_rating']].head(10))

,movieId,title,rating_count,rating_mean,weighted_rating
315,318,"Shawshank Redemption, The (1994)",63366,4.446990,4.420466
843,858,"Godfather, The (1972)",41355,4.364732,4.327039
49,50,"Usual Suspects, The (1995)",47006,4.334372,4.301909
523,527,Schindler's List (1993),50054,4.310175,4.280253
1195,1221,"Godfather: Part II, The (1974)",27398,4.275641,4.223672
1169,1193,One Flew Over the Cuckoo's Nest (1975),29932,4.248079,4.201478
895,912,Casablanca (1942),24349,4.258327,4.201063
2873,2959,Fight Club (1999),40106,4.227123,4.192632
887,904,Rear Window (1954),17449,4.271334,4.192097
737,750,Dr. Strangelove or: How I Learned to Stop Worr...,23220,4.247287,4.187975


In [10]:
def popularity_recommender(top_n=10):
    return qualified[['movieId', 'title', 'rating_count', 'rating_mean', 'weighted_rating']].head(top_n)

In [11]:
popularity_recommender(10)


,movieId,title,rating_count,rating_mean,weighted_rating
315,318,"Shawshank Redemption, The (1994)",63366,4.446990,4.420466
843,858,"Godfather, The (1972)",41355,4.364732,4.327039
49,50,"Usual Suspects, The (1995)",47006,4.334372,4.301909
523,527,Schindler's List (1993),50054,4.310175,4.280253
1195,1221,"Godfather: Part II, The (1974)",27398,4.275641,4.223672
1169,1193,One Flew Over the Cuckoo's Nest (1975),29932,4.248079,4.201478
895,912,Casablanca (1942),24349,4.258327,4.201063
2873,2959,Fight Club (1999),40106,4.227123,4.192632
887,904,Rear Window (1954),17449,4.271334,4.192097
737,750,Dr. Strangelove or: How I Learned to Stop Worr...,23220,4.247287,4.187975


## Popularity-Based Recommender

This method recommends movies based on global popularity.  
To avoid favoring movies with very few ratings, a weighted rating formula is used, combining:
- average movie rating
- number of ratings
- overall average rating across the dataset

# 6. Content-Based Recommender

In [12]:
tags = tags.dropna(subset=['tag'])

In [13]:
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()
tags_grouped.rename(columns={'tag': 'metadata'}, inplace=True)

display(tags_grouped.head())

,movieId,metadata
0,1,Watched computer animation Disney animated fea...
1,2,time travel adapted from:book board game child...
2,3,old people that is actually funny sequel fever...
3,4,chick flick revenge characters chick flick cha...
4,5,Diane Keaton family sequel Steve Martin weddin...


In [14]:
movies_with_tags = movies.merge(tags_grouped, on='movieId', how='left')

movies_with_tags['metadata'] = movies_with_tags['metadata'].fillna('')

display(movies_with_tags.head())

,movieId,title,genres,metadata
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Watched computer animation Disney animated fea...
1,2,Jumanji (1995),Adventure|Children|Fantasy,time travel adapted from:book board game child...
2,3,Grumpier Old Men (1995),Comedy|Romance,old people that is actually funny sequel fever...
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,chick flick revenge characters chick flick cha...
4,5,Father of the Bride Part II (1995),Comedy,Diane Keaton family sequel Steve Martin weddin...


In [15]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies_with_tags['metadata'])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (27278, 23863)


In [16]:
svd = TruncatedSVD(n_components=100, random_state=42)

latent_matrix_1 = svd.fit_transform(tfidf_matrix)

print("Latent matrix 1 shape:", latent_matrix_1.shape)

Latent matrix 1 shape: (27278, 100)


In [17]:
indices = pd.Series(movies_with_tags.index, index=movies_with_tags['title']).drop_duplicates()

def content_recommender(title, top_n=10):
    idx = indices[title]

    sim_scores = cosine_similarity(
        latent_matrix_1[idx].reshape(1, -1),
        latent_matrix_1
    )[0]

    sim_indices = sim_scores.argsort()[::-1][1:top_n+1]

    return movies_with_tags.iloc[sim_indices][['movieId', 'title']]

In [18]:
content_recommender("Fight Club (1999)", 10)

,movieId,title
16455,83184,Breathless (Ddongpari) (2009)
15563,79251,"Serbian Film, A (Srpski film) (2010)"
8267,8950,"Machinist, The (Maquinista, El) (2004)"
15373,78349,Exam (2009)
18511,92206,Hostel: Part III (2011)
9207,27134,Dark Portals: The Chronicles of Vidocq (Vidoc...
4132,4226,Memento (2000)
621,628,Primal Fear (1996)
10520,39427,Stay (2005)
22989,109742,Cheap Thrills (2013)


Due to memory constraints, we compute cosine similarity on-demand instead of precomputing the full similarity matrix.

# 7. Collaborative Filtering

In [21]:
ratings_small = ratings.sample(n=1000000, random_state=42)
print("Sampled ratings shape:", ratings_small.shape)

Sampled ratings shape: (1000000, 4)


In [23]:
movie_rating_counts = ratings_small.groupby('movieId')['rating'].count()
user_rating_counts = ratings_small.groupby('userId')['rating'].count()

active_movies = movie_rating_counts[movie_rating_counts >= 50].index
active_users = user_rating_counts[user_rating_counts >= 20].index

ratings_filtered = ratings_small[
    ratings_small['movieId'].isin(active_movies) &
    ratings_small['userId'].isin(active_users)
].copy()

print("Filtered shape:", ratings_filtered.shape)

Filtered shape: (350419, 4)


In [24]:
user_movie_matrix = ratings_filtered.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

print("User-Movie Matrix shape:", user_movie_matrix.shape)

User-Movie Matrix shape: (11524, 3176)


Due to the large size of the MovieLens 20M dataset, a subset of 1 million ratings was used for collaborative filtering to ensure computational feasibility.

In [25]:
user_movie_sparse = csr_matrix(user_movie_matrix.values)
print(user_movie_sparse.shape)

(11524, 3176)


In [26]:
svd_cf = TruncatedSVD(n_components=50, random_state=42)
latent_matrix_2 = svd_cf.fit_transform(user_movie_sparse)

print("Latent matrix 2 shape:", latent_matrix_2.shape)

Latent matrix 2 shape: (11524, 50)


In [27]:
movie_ids_cf = user_movie_matrix.columns.tolist()

movie_id_to_title = movies.set_index('movieId')['title'].to_dict()
title_to_movie_id = movies.set_index('title')['movieId'].to_dict()

In [28]:
movie_factors = svd_cf.components_.T
print("Movie factors shape:", movie_factors.shape)

Movie factors shape: (3176, 50)


In [29]:
def collaborative_recommender(title, top_n=10):
    if title not in title_to_movie_id:
        return "Movie not found in dataset."

    movie_id = title_to_movie_id[title]

    if movie_id not in movie_ids_cf:
        return "Movie not available in filtered collaborative matrix."

    idx = movie_ids_cf.index(movie_id)

    sim_scores = cosine_similarity(
        movie_factors[idx].reshape(1, -1),
        movie_factors
    )[0]

    sim_indices = sim_scores.argsort()[::-1][1:top_n+1]
    recommended_movie_ids = [movie_ids_cf[i] for i in sim_indices]

    result = movies[movies['movieId'].isin(recommended_movie_ids)][['movieId', 'title']]
    return result

In [30]:
collaborative_recommender("Fight Club (1999)", 10)

,movieId,title
145,147,"Basketball Diaries, The (1995)"
423,427,Boxing Helena (1993)
1015,1034,Freeway (1996)
3409,3499,Misery (1990)
4543,4638,Jurassic Park III (2001)
4606,4701,Rush Hour 2 (2001)
5857,5956,Gangs of New York (2002)
7259,7371,Dogville (2003)
10974,44972,Scary Movie 4 (2006)
16635,84152,Limitless (2011)


## Collaborative Filtering

A user-movie rating matrix is constructed from the ratings data.  
To make computation feasible on MovieLens 1M, only active users and frequently rated movies are retained.  
TruncatedSVD is then applied to generate a latent representation (Latent Matrix 2), and movie-to-movie similarity is computed in the latent space.

In [31]:
collaborative_recommender("Shawshank Redemption, The (1994)", 10)

,movieId,title
189,191,"Scarlet Letter, The (1995)"
237,240,Hideaway (1995)
312,315,"Specialist, The (1994)"
339,343,"Baby-Sitters Club, The (1995)"
729,742,Thinner (1996)
986,1005,D3: The Mighty Ducks (1996)
1163,1187,Passion Fish (1992)
1956,2040,"Computer Wore Tennis Shoes, The (1969)"
4637,4733,"Curse of the Jade Scorpion, The (2001)"
7787,8387,Police Academy: Mission to Moscow (1994)


# 8. Matrix Factorization (Surprise)

In [32]:
ratings_surprise = ratings.sample(n=1000000, random_state=42).copy()
print(ratings_surprise.shape)
display(ratings_surprise.head())

(1000000, 4)


,userId,movieId,rating,timestamp
17679788,122270,8360,3.5,1335056824
7106385,49018,32,2.0,1000194636
12970708,89527,109374,3.5,1420536400
15426752,106704,1060,3.0,948576477
6934678,47791,1732,2.0,1137685703


In [33]:
ratings_surprise = ratings.sample(n=1000000, random_state=42).copy()
print(ratings_surprise.shape)
display(ratings_surprise.head())

(1000000, 4)


,userId,movieId,rating,timestamp
17679788,122270,8360,3.5,1335056824
7106385,49018,32,2.0,1000194636
12970708,89527,109374,3.5,1420536400
15426752,106704,1060,3.0,948576477
6934678,47791,1732,2.0,1137685703


In [35]:
reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings_surprise[['userId', 'movieId', 'rating']],
    reader
)

In [36]:
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

In [37]:
svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

In [38]:
predictions = svd_model.test(testset)

print("RMSE:", accuracy.rmse(predictions))
print("MAE:", accuracy.mae(predictions))

RMSE: 0.9029
RMSE: 0.9029169056161181
MAE:  0.6978
MAE: 0.6978029454365189


In [39]:
def predict_rating(user_id, movie_title):
    matched = movies[movies['title'] == movie_title]

    if matched.empty:
        return "Movie not found."

    movie_id = matched.iloc[0]['movieId']
    pred = svd_model.predict(user_id, movie_id)

    return {
        'userId': user_id,
        'movieId': movie_id,
        'title': movie_title,
        'estimated_rating': pred.est
    }

In [40]:
predict_rating(1, "Fight Club (1999)")

{'userId': 1,
 'movieId': 2959,
 'title': 'Fight Club (1999)',
 'estimated_rating': 4.329074520319652}

## Matrix Factorization using Surprise

To implement matrix factorization, the Surprise library was used with the SVD algorithm.  
A subset of the MovieLens 20M ratings data was used for training and evaluation.  
The model predicts unseen user-movie ratings by learning latent factors for users and movies.

# 9. Hybrid Model

In [41]:
def get_content_candidates(title, top_n=20):
    idx = indices[title]

    sim_scores = cosine_similarity(
        latent_matrix_1[idx].reshape(1, -1),
        latent_matrix_1
    )[0]

    sim_indices = sim_scores.argsort()[::-1][1:top_n+1]

    candidates = movies_with_tags.iloc[sim_indices][['movieId', 'title']].copy()
    candidates['content_score'] = sim_scores[sim_indices]

    return candidates

In [42]:
def hybrid_recommender(user_id, title, top_n=10):
    candidates = get_content_candidates(title, top_n=30).copy()

    estimated_ratings = []

    for movie_id in candidates['movieId']:
        pred = svd_model.predict(user_id, movie_id)
        estimated_ratings.append(pred.est)

    candidates['predicted_rating'] = estimated_ratings

    # ترکیب content score و predicted rating
    candidates['hybrid_score'] = (
        0.5 * candidates['content_score'] +
        0.5 * (candidates['predicted_rating'] / 5.0)
    )

    candidates = candidates.sort_values('hybrid_score', ascending=False)

    return candidates[['movieId', 'title', 'content_score', 'predicted_rating', 'hybrid_score']].head(top_n)

In [43]:
hybrid_recommender(1, "Fight Club (1999)", 10)

,movieId,title,content_score,predicted_rating,hybrid_score
15563,79251,"Serbian Film, A (Srpski film) (2010)",0.788227,3.907222,0.784836
9476,27773,Old Boy (2003),0.697278,4.327817,0.781421
4132,4226,Memento (2000),0.723990,4.156914,0.777687
2244,2329,American History X (1998),0.668498,4.426569,0.776906
8267,8950,"Machinist, The (Maquinista, El) (2004)",0.769410,3.780828,0.762788
11401,48780,"Prestige, The (2006)",0.707904,4.032720,0.757224
16455,83184,Breathless (Ddongpari) (2009),0.788566,3.618622,0.756145
3917,4011,Snatch (2000),0.688152,4.078504,0.751926
621,628,Primal Fear (1996),0.723655,3.871977,0.749025
2509,2594,Open Your Eyes (Abre los ojos) (1997),0.685633,3.980946,0.740911


## Hybrid Recommender

The hybrid recommender combines:
- content-based similarity from TF-IDF + TruncatedSVD
- personalized rating prediction from Surprise SVD

First, content-based filtering is used to retrieve candidate movies similar to the input movie.  
Then, the Surprise SVD model predicts the target user's rating for each candidate movie.  
Finally, both scores are combined into a hybrid score for ranking.

# 10. Conclusion

In this notebook, multiple recommender system approaches were implemented on the MovieLens dataset:

1. Popularity-Based Recommendation
2. Content-Based Filtering using TF-IDF and TruncatedSVD
3. Collaborative Filtering using User-Movie Matrix and Latent Factors
4. Matrix Factorization using Surprise SVD
5. Hybrid Recommendation combining content similarity and predicted ratings

Because MovieLens 20M is computationally intensive, subsets of the ratings data were used in collaborative and matrix factorization stages to ensure feasibility in Google Colab.